# Model Experimentation

### Experiments:
-"distilbert-base-multilingual-cased" (base) -> 20% head tunned
-"distilbert-base-multilingual-cased" (LORA) -> add and train new layer
-"microsoft/mdeberta-v3-base" (base) -> 20% head tunned
-"microsoft/mdeberta-v3-base" (LORA) -> add and train new layer


In [2]:
# ----import tools----

#utilities
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import datasets

#text processing
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    TrainingArguments,
    Trainer
)

#LORA Fineting
from peft import LoraConfig, get_peft_model

# Tensorflow and PyTorch
import tensorflow as tf

# CollumnTransformer and Pipeline for preprocessing
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

#evaluation
import evaluate

# MLflow
import mlflow

# I. Preparing Data

In [40]:
# Load Data
training_set_path = "../data/processed/train_ds/data-00000-of-00001.arrow"
ds = datasets.Dataset.from_file(training_set_path)
ds.to_pandas().head(10)

,text,labels
0,the original online source for market research...,fraud
1,last up to 4 hours with 100 % natural viagra !...,fraud
2,doctor discovers s ' perm increasement pill\n\...,fraud
3,good time after party : )\n\nthe latest invent...,fraud
4,Important notice: Your account verification re...,fraud
5,Claim a free beauty product! today and enjoy e...,ham
6,FROM 88066 LOST £12 HELP\n,fraud
7,business relationship\n\ni am engineer mr duke...,fraud
8,rolex watches now for pea nut\n\nhows it been ...,fraud
9,"caller: Good afternoon, your catering order fo...",ham


In [41]:
ds.to_pandas().describe()

,text,labels
count,14786,14786
unique,14786,2
top,the original online source for market research...,ham
freq,1,7412


In [42]:
ds.to_pandas().info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14786 entries, 0 to 14785
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    14786 non-null  object
 1   labels  14786 non-null  object
dtypes: object(2)
memory usage: 231.2+ KB


In [43]:
ds.to_pandas()['labels'].value_counts()

labels
ham      7412
fraud    7374
Name: count, dtype: int64

In [44]:
# train test split
ds_split = ds.train_test_split(test_size=0.1, seed=42)

ds_train = ds_split['train']
ds_val = ds_split['test']

print(f"Train Set: {len(ds_train)}, {ds_train.to_pandas()['labels'].value_counts()}")
print(f"Validation Set: {len(ds_val)}, {ds_val.to_pandas()['labels'].value_counts()}")

Train Set: 13307, labels
ham      6704
fraud    6603
Name: count, dtype: int64
Validation Set: 1479, labels
fraud    771
ham      708
Name: count, dtype: int64


In [45]:
lengths = [len(x.split()) for x in ds_train["text"]]
print(f"90th percentile length: {np.percentile(lengths, 90)}")

90th percentile length: 222.0


# Preprocess data for distilbert-base-multilingual

In [50]:
# Decodning {"Fraud":1, "Ham":0}
str2int = {"fraud":1, "ham":0}

encoded_ds_train = ds_train.map(lambda x: {"labels": str2int[x["labels"]]})
encoded_ds_val = ds_val.map(lambda x: {"labels": str2int[x["labels"]]})

# Tokenization for distilbert
distilbert_model_id = "distilbert-base-multilingual-cased"

tokenizer = AutoTokenizer.from_pretrained(distilbert_model_id)

def tokenize_function(examples):
    return tokenizer(examples["text"],
                      padding="max_length", 
                      truncation=True,
                      max_length=256)

tokenized_ds_train = encoded_ds_train.map(tokenize_function, batched=True)
tokenized_ds_val = encoded_ds_val.map(tokenize_function, batched=True)

# Change int64 -> 32 for faster trainning
downsized_features = datasets.Features({
    'text': datasets.Value('string'),
    'labels': datasets.Value('int32'), 
    'input_ids': datasets.Sequence(datasets.Value('int32')),
    'attention_mask': datasets.Sequence(datasets.Value('int8')),
    'token_type_ids': datasets.Sequence(datasets.Value('int8'))
})

# Cast the datasets to these strict features
tokenized_ds_train = tokenized_ds_train.cast(downsized_features)
tokenized_ds_val = tokenized_ds_val.cast(downsized_features)

print(tokenized_ds_train.features)
print(f"Train Set:{tokenized_ds_train}\n")

print(tokenized_ds_val.features)
print(f"Validation Set:{tokenized_ds_val}\n")

{'text': Value('string'), 'labels': Value('int32'), 'input_ids': List(Value('int32')), 'attention_mask': List(Value('int8')), 'token_type_ids': List(Value('int8'))}
Train Set:Dataset({
    features: ['text', 'labels', 'input_ids', 'attention_mask', 'token_type_ids'],
    num_rows: 13307
})

{'text': Value('string'), 'labels': Value('int32'), 'input_ids': List(Value('int32')), 'attention_mask': List(Value('int8')), 'token_type_ids': List(Value('int8'))}
Validation Set:Dataset({
    features: ['text', 'labels', 'input_ids', 'attention_mask', 'token_type_ids'],
    num_rows: 1479
})



In [51]:
print(tokenized_ds_train.features["labels"])

Value('int32')


# Fine-tuning DistilBERT with LoRa

In [55]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
distilbert_model_id = "distilbert-base-multilingual-cased"

bert_base = AutoModelForSequenceClassification.from_pretrained(
    distilbert_model_id, num_labels=2)

# apply LORA finetuning
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_lin", "k_lin", "v_lin", "out_lin"],
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_CLS",
)

# get the LORA model
bert_lora = get_peft_model(bert_base, lora_config)
bert_lora.print_trainable_parameters()

# set evaluation metric for LORA finetuning
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predicted_class = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predicted_class)
    precision = precision_score(labels, predicted_class)
    recall = recall_score(labels, predicted_class)
    f1 = f1_score(labels, predicted_class)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
    }

# Set LORA model for training
training_lora_distilmBERT_args  = TrainingArguments(
    bf16=True,
    fp16=False,
    output_dir="./results_lora_distil-mBert",
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    save_strategy="best", # save the model at the end of each epoch
    load_best_model_at_end=True, # load the best model at the end of training
    metric_for_best_model='recall' # use recall to evaluate the best model
)

# 
trainer_lora_distilmBERT = Trainer(
    model=bert_lora,
    args=training_lora_distilmBERT_args,
    train_dataset=tokenized_ds_train,
    eval_dataset=tokenized_ds_val,
    compute_metrics=compute_metrics
)

trainer_lora_distilmBERT.train()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 887,042 || all params: 136,213,252 || trainable%: 0.6512


/opt/miniconda3/envs/dlenv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
500,0.294989
1000,0.155425
1500,0.108290
2000,0.089680
2500,0.088402
3000,0.075062


TrainOutput(global_step=3328, training_loss=0.12965406477451324, metrics={'train_runtime': 9168.6455, 'train_samples_per_second': 5.805, 'train_steps_per_second': 0.363, 'total_flos': 3598010312171520.0, 'train_loss': 0.12965406477451324, 'epoch': 4.0})

In [56]:
lora_distilmBERT_results = trainer_lora_distilmBERT.evaluate()

Training Loss,Validation Loss,Step,Accuracy,Precision,Recall,F1 Score
0.075062,0.099909,3328,0.968222,0.977573,0.961089,0.969261


In [58]:
print("LORA Finetuning Results:")
print(lora_distilmBERT_results)

LORA Finetuning Results:
{'eval_loss': 0.09990882128477097, 'eval_accuracy': 0.9682217714672076, 'eval_precision': 0.9775725593667546, 'eval_recall': 0.9610894941634242, 'eval_f1_score': 0.9692609548724657}


### Logging Model to ML Flow

In [59]:
import mlflow
import mlflow.transformers
from mlflow.tracking import MlflowClient

mlflow.set_tracking_uri("http://localhost:8080")
mlflow.set_experiment("scam-detector-finetuning")

with mlflow.start_run(run_name="lora_distilmBERT_v2") as run:
        # 1. Log params
        mlflow.log_params(training_lora_distilmBERT_args.to_dict())
            
        # 2. Log metrics
        mlflow.log_metrics(lora_distilmBERT_results)

        # 3. Log model + tokenizer
        mlflow.transformers.log_model(
        transformers_model={
                    "model":     trainer_lora_distilmBERT.model,       # model 
                    "tokenizer": tokenizer # tokenizer 
                },
                artifact_path="lora_distilmBERT_v2",
                task="text-classification",
            )

# Verifikasi model yang sudah di log
client = MlflowClient(tracking_uri="http://localhost:8080")
experiment = client.get_experiment_by_name("scam-detector-finetuning")
runs = client.search_runs(experiment_ids=[experiment.experiment_id])
print(f"Runs in experiment 'scam-detector-finetuning': {[run.info.run_id for run in runs]}")


python(59284) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(59285) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
2026/05/09 18:22:47 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/09 18:22:47 INFO mlflow.transformers: Overriding save_pretrained to False for PEFT models, following the Transformers behavior. The PEFT adaptor and config will be saved, but the base model weights will not and reference to the HuggingFace Hub repository will be logged instead.
2026/05/09 18:22:51 INFO mlflow.transformers: Skipping saving pretrained model weights to disk as the save_pretrained argumentis set to False. The reference to the HuggingFace Hub repository distilbert-base-multilingual-cased will be logged instead.
2026/05/09 18:22:52 INFO mlflow.transformers: A local checkpoint path or PEFT model is given as the `transformers_model`. To avoid loading the full model into m

🏃 View run lora_distilmBERT_v2 at: http://localhost:8080/#/experiments/1/runs/bf1a1dd415d741baa0783d07e4dbaca6
🧪 View experiment at: http://localhost:8080/#/experiments/1
Runs in experiment 'scam-detector-finetuning': ['bf1a1dd415d741baa0783d07e4dbaca6', '4b367643a71244478d7946e4f7685d2d', '0783e27cccd94747bec51d71e5731cd8', '149c470308ad4d4ea886051b09346fd6', 'd7f0b29667e94ccb99f8bb5e7050d9a8', 'f3315e08d68f45a583bd77f0959a4f50', 'db87fe9108a944e3a171d2468e2764c6', '7a2682476fab4ad49d8dac6a1901874a']
